In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from pyspark.sql.window import Window

In [0]:
%sql
select * from fintech360.bronze.emi_data;

In [0]:
df = spark.read.table('fintech360.bronze.emi_data')

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DecimalType, IntegerType, LongType, TimestampType
)

EMI_PARENT_SCHEMA = StructType([
    StructField("emi_id", StringType(), True),
    StructField("loan_id", StringType(), True),
    StructField("installment_number", IntegerType(), True),
    StructField("due_date", DateType(), True),
    StructField("emi_amount", DecimalType(18, 2), True),
    StructField("paid_amount", DecimalType(18, 2), True),
    StructField("paid_at", DateType(), True),
    StructField("emi_status", StringType(), True),
    StructField("days_past_due", IntegerType(), True),
    StructField("payment_method", StringType(), True),
    StructField("late_fee_amount", DecimalType(18, 2), True),
    StructField("created_at", LongType(), True),
    StructField("updated_at", LongType(), True),
    StructField("emi_details_json", StringType(), True),
    StructField("_rescued_data", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True)
])

In [0]:
from pyspark.sql import functions as F

def normalize_emi_schema(df, parent_schema=EMI_PARENT_SCHEMA):
    parent_names = [field.name for field in parent_schema.fields]
    parent_fields = {field.name: field for field in parent_schema.fields}

    # Capture unexpected columns
    extra_cols = [c for c in df.columns if c not in parent_names]

    if extra_cols:
        extra_json = F.to_json(
            F.struct(*[F.col(c) for c in extra_cols])
        )

        if "_rescued_data" in df.columns:
            df = df.withColumn(
                "_rescued_data",
                F.when(
                    F.col("_rescued_data").isNull(),
                    extra_json
                ).otherwise(
                    F.concat_ws("\n", F.col("_rescued_data"), extra_json)
                )
            )
        else:
            df = df.withColumn("_rescued_data", extra_json)

    # Normalize identifiers such as 1001.0 to 1001
    for col_name in ["emi_id", "loan_id"]:
        if col_name in df.columns:
            value = F.trim(F.col(col_name).cast("string"))
            numeric_value = value.cast("decimal(38,10)")

            df = df.withColumn(
                col_name,
                F.when(
                    value.rlike(r"^[+-]?\d+\.0+$"),
                    numeric_value.cast("decimal(38,0)").cast("string")
                ).otherwise(value)
            )

    # Add missing fields and cast to canonical types
    for name, field in parent_fields.items():
        if name not in df.columns:
            df = df.withColumn(name, F.lit(None).cast(field.dataType))
        else:
            df = df.withColumn(name, F.col(name).cast(field.dataType))

    return df.select(*parent_names)

In [0]:
df = normalize_emi_schema(df)

df.printSchema()
display(df.limit(10))

In [0]:
(df.write
    .mode('append')
    .format('delta')
    .saveAsTable('fintech360.schema_validation.emi_data'))